In [30]:
# get data 
import pandas as pd
DATA = "MultiRC/"
df = pd.read_json(DATA + 'train_456-fixedIds.json')
test_data = pd.read_json(DATA + 'test_83-fixedIds.json')
print(df.head())


                                                data
0  {'paragraph': {'text': '<b>Sent 1: </b>Animate...
1  {'paragraph': {'text': '<b>Sent 1: </b>Before ...
2  {'paragraph': {'text': '<b>Sent 1: </b>Hotel C...
3  {'paragraph': {'text': '<b>Sent 1: </b>Elaan i...
4  {'paragraph': {'text': '<b>Sent 1: </b>Not unt...


In [9]:

first_row_df = df.iloc[0]
paragraph = first_row_df.iloc[0]['paragraph']

#text 
paragraph_text = paragraph['text']
paragraph_questions = paragraph['questions']
print(paragraph_text)
print(paragraph_questions)

<b>Sent 1: </b>Animated history of the US.<br><b>Sent 2: </b>Of course the cartoon is highly oversimplified, and most critics consider it one of the weakest parts of the film.<br><b>Sent 3: </b>But it makes a valid claim which you ignore entirely: That the strategy to promote "gun rights" for white people and to outlaw gun possession by black people was a way to uphold racism without letting an openly terrorist organization like the KKK flourish.<br><b>Sent 4: </b>Did the 19th century NRA in the southern states promote gun rights for black people?<br><b>Sent 5: </b>I highly doubt it.<br><b>Sent 6: </b>But if they didn't, one of their functions was to continue the racism of the KKK.<br><b>Sent 7: </b>This is the key message of this part of the animation, which is again being ignored by its critics.<br><b>Sent 8: </b>Buell shooting in Flint.<br><b>Sent 9: </b>You write: "Fact: The little boy was the class thug, already suspended from school for stabbing another kid with a pencil, and had

In [ ]:
#first question + answer
first_question = paragraph_questions[0]
first_question_text = first_question['question']
first_answer = first_question['answers'][0]
answer = first_answer['text']
result = first_answer['isAnswer']

"""
the dataset is a bit weird so we have to plug in the paragrapg, question and answer
and then the model will predict if the answer is correct or not 
"""
print(paragraph_text)
print(first_question_text)
print(answer)
print(result)

<b>Sent 1: </b>Animated history of the US.<br><b>Sent 2: </b>Of course the cartoon is highly oversimplified, and most critics consider it one of the weakest parts of the film.<br><b>Sent 3: </b>But it makes a valid claim which you ignore entirely: That the strategy to promote "gun rights" for white people and to outlaw gun possession by black people was a way to uphold racism without letting an openly terrorist organization like the KKK flourish.<br><b>Sent 4: </b>Did the 19th century NRA in the southern states promote gun rights for black people?<br><b>Sent 5: </b>I highly doubt it.<br><b>Sent 6: </b>But if they didn't, one of their functions was to continue the racism of the KKK.<br><b>Sent 7: </b>This is the key message of this part of the animation, which is again being ignored by its critics.<br><b>Sent 8: </b>Buell shooting in Flint.<br><b>Sent 9: </b>You write: "Fact: The little boy was the class thug, already suspended from school for stabbing another kid with a pencil, and had

In [ ]:
# full pipeline ------------ this took me way too long 
def get_paragraph_and_questions(df):
    for row in df.itertuples():
        para_id = row[0]
        paragraph = row[1]['paragraph']
        paragraph_text = paragraph['text']
        paragraph_questions = paragraph['questions']
        for question in paragraph_questions:
            question_text = question['question']
            for answer in question['answers']:
                answer_text = answer['text']
                is_answer = answer['isAnswer']
                yield paragraph_text, question_text, answer_text, is_answer

import re

def clean_paragraph(input_tuple):
    text = input_tuple[0] 
    text = re.sub(r"<b>Sent \d+: </b>", "", text)  # remove "Sent N:" markers
    text = text.replace("<br>", " ")                # line breaks -> spaces
    text = re.sub(r"\s+", " ", text).strip()        # tidy whitespace
    input_tuple = (text,) + input_tuple[1:]  
    return input_tuple

# to use u iniitialize the generator
questions_answers = get_paragraph_and_questions(df)

# then use next everytime you want to get the next value (question answer pairs)

print(clean_paragraph(next(questions_answers))) # Get the first item


for _ in range(150):
    next(questions_answers)

# Print the 103rd item
print(clean_paragraph(next(questions_answers)))



('Animated history of the US. Of course the cartoon is highly oversimplified, and most critics consider it one of the weakest parts of the film. But it makes a valid claim which you ignore entirely: That the strategy to promote "gun rights" for white people and to outlaw gun possession by black people was a way to uphold racism without letting an openly terrorist organization like the KKK flourish. Did the 19th century NRA in the southern states promote gun rights for black people? I highly doubt it. But if they didn\'t, one of their functions was to continue the racism of the KKK. This is the key message of this part of the animation, which is again being ignored by its critics. Buell shooting in Flint. You write: "Fact: The little boy was the class thug, already suspended from school for stabbing another kid with a pencil, and had fought with Kayla the day before". This characterization of a six-year-old as a pencil-stabbing thug is exactly the kind of hysteria that Moore\'s film war

In [31]:
test = get_paragraph_and_questions(test_data)
count =0
while True:
    try:
        item = next(test)
        count += 1
    except StopIteration:
        break
print(f"Total items processed: {count}")

Total items processed: 4848
